# Perceiver CPI: train 3 models (Davis, Metz, KIBA)

This notebook trains **one model for each of three datasets**, for a student project that reproduces *Perceiver CPI* (Nguyen et al., *Bioinformatics* 2023).

The model is small, so the speed depends mostly on the runtime's CPU cores:
- **Powerful runtime (6 or more CPU cores, e.g. A100 / L4 / H100):** the three models train **at the same time**, roughly 5–10 hours in total. With several GPUs, each model gets its own.
- **Basic runtime (e.g. T4 with 2 cores):** they train one after another, roughly 15–20 hours.

Cell 5 measures the real speed and prints an estimate.

**How to run it**
1. **Runtime → Change runtime type → GPU** (the most powerful one available), then **Save**.
2. **Runtime → Run all.** When asked, allow access to Google Drive.
3. Leave it running.
   - Everything is saved to the `runs` folder next to this notebook on Google Drive, after every epoch.
   - **If the session ends or disconnects, just choose Runtime → Run all again.** Finished datasets are skipped, and the one in progress continues from its last finished epoch.
4. When the last cell prints **ALL DONE**, the file `results_to_copy.zip` is in the same Drive folder. Please let the student know.

**If this notebook was shared with you:** first add the shared folder to your own Drive. Go to drive.google.com → *Shared with me* → right-click the folder `perceiver_cpi_colab` → *Organize* → *Add shortcut* → *My Drive*. Colab can only see folders in your own Drive.

Nothing else needs to be set up. The notebook installs the tested package versions with pip and uses Colab's own PyTorch. pip may print warnings about version conflicts with other Colab packages; these are harmless here.

In [ ]:
# 1. Settings and Google Drive (normally nothing to change here)
import glob
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

DRIVE_FOLDER_NAME = 'perceiver_cpi_colab'   # the Drive folder holding this notebook and the zip below
BUNDLE_ZIP = 'perceiver_gpu_bundle.zip'     # code + data (fold 0 of the novel-pair split)
DATASETS = ['davis', 'metz', 'kiba']
TEST_ARGS = os.environ.get('PERCEIVER_TEST_ARGS', '')  # empty on Colab; only used to test this notebook offline

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB, LOCAL = True, Path('/content')
except ImportError:
    IN_COLAB, LOCAL = False, Path(os.environ['PERCEIVER_TEST_LOCAL'])
print('Running on Google Colab.' if IN_COLAB else 'Running an offline test (not Colab).')

In [ ]:
# 2. Find the project folder on Google Drive; results will be saved there
def find_drive_folder():
    if not IN_COLAB:
        return Path(os.environ['PERCEIVER_TEST_DRIVE'])
    patterns = [f'/content/drive/MyDrive/{DRIVE_FOLDER_NAME}/{BUNDLE_ZIP}',
                f'/content/drive/MyDrive/*/{DRIVE_FOLDER_NAME}/{BUNDLE_ZIP}',
                f'/content/drive/Shareddrives/*/{DRIVE_FOLDER_NAME}/{BUNDLE_ZIP}',
                f'/content/drive/MyDrive/*/{BUNDLE_ZIP}',
                f'/content/drive/MyDrive/*/*/{BUNDLE_ZIP}']
    for pattern in patterns:
        found = sorted(glob.glob(pattern))
        if found:
            return Path(found[0]).parent
    raise FileNotFoundError(
        f'Could not find the folder "{DRIVE_FOLDER_NAME}" (with {BUNDLE_ZIP}) in your Google Drive.\n'
        'If it was shared with you: drive.google.com -> "Shared with me" -> right-click the folder -> '
        '"Organize" -> "Add shortcut" -> "My Drive". Then choose Runtime -> Run all again.')

DRIVE_DIR = find_drive_folder()
RUNS = DRIVE_DIR / 'runs'
os.environ['PERCEIVER_RUNS'] = str(RUNS)  # train_all.py and pack_results.py save here
print('Project folder:', DRIVE_DIR)
print('Results go to: ', RUNS)

In [ ]:
# 3. Unpack the code and data onto Colab's fast local disk (takes a few seconds)
WORK = LOCAL / 'perceiver_gpu'
if WORK.exists():
    shutil.rmtree(WORK)
with zipfile.ZipFile(DRIVE_DIR / BUNDLE_ZIP) as zf:
    zf.extractall(LOCAL)
print((WORK / 'BUNDLE_INFO.txt').read_text())

In [ ]:
# 4. Show the machine and install the packages (1-3 minutes)
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'CPU cores: {os.cpu_count()}')

def pip_install(*args):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], capture_output=True, text=True)

print('Installing the tested package versions...')
result = pip_install('-r', str(WORK / 'model_2d' / 'requirements.txt'))
if result.returncode != 0:
    print(result.stdout[-3000:], result.stderr[-3000:])
    print('The exact tested versions could not be installed here; installing current versions instead.')
    result = pip_install('rdkit', 'tape-proteins', 'typed-argument-parser', 'tensorboardX', 'lifelines', 'hyperopt')
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError('Installing packages failed (see above). Please send this output to the student.')
print('Packages installed. PyTorch (Colab\'s own):')
print(subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, "| GPU:", '
                      'torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")'],
                     capture_output=True, text=True).stdout)

In [ ]:
# 5. Check that everything works: data files, packages, GPU, and a short test training run.
#    The first time, it also trains 1 full epoch to measure the speed (about 5 minutes) and prints an estimate.
started_before = any((RUNS / d).exists() for d in DATASETS)
print('Checking the setup' + (' (resuming, so no timing test)...' if started_before else ' (about 5 minutes)...'))
command = [sys.executable, 'check_setup.py', *TEST_ARGS.split()] + (['--skip-timing'] if started_before else [])
result = subprocess.run(command, cwd=WORK, capture_output=True, text=True)
print(result.stdout, result.stderr[-3000:])
if result.returncode != 0:
    raise RuntimeError('The setup check failed (see above). Please send this output to the student.')

In [ ]:
# 6. Train Davis, Metz and KIBA (several hours).
#    Same time (6+ CPU cores): a progress line every 5 minutes, e.g. "kiba epoch 37/150 (~6.1 h left)".
#    One after another (fewer cores): a progress bar per dataset, epochs done/150 [time so far < time left].
#    If Colab disconnects: Runtime -> Run all again; each dataset continues from its last finished epoch.
os.chdir(WORK)
get_ipython().system(f'"{sys.executable}" train_all.py {TEST_ARGS}')
not_finished = [d for d in DATASETS if not (RUNS / d / 'test_scores.csv').exists()]
if not_finished:
    raise RuntimeError(f'Not finished yet: {", ".join(not_finished)}. Choose Runtime -> Run all to continue.')

In [ ]:
# 7. Pack the results into results_to_copy.zip (in the Drive folder) and show the test scores
result = subprocess.run([sys.executable, 'pack_results.py'], cwd=WORK, capture_output=True, text=True)
print(result.stdout, result.stderr[-2000:])
for d in DATASETS:
    print(f'--- {d} (novel pair, fold 0) ---')
    print((RUNS / d / 'test_scores.csv').read_text())
print('ALL DONE. results_to_copy.zip is in the Drive folder:', DRIVE_DIR)

**Done.** Please tell the student that training has finished. The file `results_to_copy.zip` (about 55 MB) and the `runs` folder are in the shared Drive folder. Thank you!